================================================================================================================
The following is to calculate first component of depression residuals to non-sleep-related depression for **GP1**
================================================================================================================
The training and test sets are the same from GLM
================================================================================================================
Age and sex effects have already been removed in GLM setp
================================================================================================================

In [ ]:
import sys
sys.path.append('working_path') 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.model_selection import KFold

from sklearn.preprocessing import StandardScaler
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity

In [ ]:
# Initiate a dictionary
dataframes_train = {}
dataframes_test = {}

# Loop the dataframes and store in the dictionary
for i in range(1, 6):
    path_train = f'save_path/depression_residuals_training_fold_{i}.csv'
    path_test = f'save_path/depression_residuals_test_fold_{i}.csv'

    dataframes_train[f'df_depr_res_train_{i}'] = pd.read_csv(path_train)
    dataframes_test[f'df_depr_res_test_{i}'] = pd.read_csv(path_test)

# Get the global info for all dfs
globals().update(dataframes_train)
globals().update(dataframes_test)

In [ ]:
# Define original variables
N_12 = ['n_12_depression_variables']
RDS_4 = ['rds4_depression_variables']

res_cols = [f"{col}_res" for col in (N_12 + RDS_4)]

# Define scaling
scale = StandardScaler()

In [ ]:
# Loop through all dfs
for fold in range(1, 6):
    df_res_test = globals()[f'df_depr_res_test_{fold}']
    df_depr_res_train = globals()[f'df_depr_res_train_{fold}'][res_cols]
    df_depr_res_test = globals()[f'df_depr_res_test_{fold}'][res_cols]

    # Scaling on training set and project to test set
    df_depr_res_train_s = scale.fit_transform(df_depr_res_train)
    df_depr_res_test_s = scale.transform(df_depr_res_test)
    
    # Run PCA to get eigenvalues
    fa = FactorAnalyzer(rotation='varimax', method='principal')
    fa.fit(df_depr_res_train_s)
    ev_depression, v_depression = fa.get_eigenvalues()

    # Determine number of components with eigenvalue > 1
    n_components_depression = sum(ev_depression > 1)
    print(f"Number of components for depression with eigenvalue > 1 in fold {fold}: {n_components_depression}")

    # Get rotated loadings for depression
    loadings_depression = pd.DataFrame(fa.loadings_, index=res_cols)
    print(f"Rotated component loadings for depression in fold {fold}:\n", loadings_depression)

    # Get variance explained by each component for depression
    variance_depression = fa.get_factor_variance()
    explained_variance_depression_df = pd.DataFrame({
        'SS Loadings': variance_depression[0],
        'Proportion Var': variance_depression[1],
        'Cumulative Var': variance_depression[2]
    }, index=[f'PC{i+1}' for i in range(len(variance_depression[0]))])
    print("\nExplained Variance:\n", explained_variance_depression_df)

    # Transform to ML data
    depression_scores = fa.transform(df_depr_res_test_s)
    
    # Plot the loadings for depression
    plt.figure(figsize=(10, 6))
    sns.heatmap(loadings_depression, annot=True, cmap='viridis', cbar=True, center=0)
    plt.title(f'Rotated Component Loadings for depression in fold {fold}')
    plt.xlabel('Components')
    plt.ylabel('Depression')
    plt.tight_layout()
    plt.show()

    # Only save PC1 to the test set
    df_PC1 = df_res_test.copy()
    df_PC1['pca_comp1_depr'] = depression_scores[:, 0]
    df_PC1.to_csv(f'working_path/PCA_PC1_depression_residuals_fold_{fold}.csv', index=False)
